# 0.12 — GenAI burst window: BERTrend (Nov 2022 – Jan 2023)

Follow-up to [`0.11`](0.11-genai-keyword-timeline.ipynb): keyword share ramps from **Dec 2022** (ChatGPT **2022-11-30**). Can unsupervised BERTrend find a genAI theme when the signal is concentrated?

**Window:** 2022-11-01 → 2023-01-31 (Nov · Dec · Jan)

Only **~94** STANDARD lexicon hits in this window (Nov **0**, Dec 16, Jan 78) — lexicon-scoped BERTrend is a **small validation run**; all-news is the main unsupervised test.

**Experiments:** all-news (unsupervised) + lexicon-scoped (validation subcorpus).
**Granularities:** 7d and 14d (more slices than 21d in a 3-month window)

**Post-hoc check:** genAI keyword overlap on discovered themes (not used in clustering).

**Milestones:** ChatGPT 2022-11-30 · CHAT ETF 2023-05-17

**Outputs:** `scan_genai_burst_{all,lex}_{7,14}d.parquet`, `genai_burst_intensity.parquet`, `genai_burst_compare.parquet`

In [1]:
import os, re, sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from loguru import logger as _lg
_lg.remove(); _lg.add(sys.stderr, level="WARNING")

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("BERTREND_BASE_DIR", str(_ROOT / "notebooks" / "output" / "bertrend_base"))
RAW_DIR = _ROOT / "data" / "raw"
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic.representation import MaximalMarginalRelevance
from bertrend.BERTrend import BERTrend
from bertrend.BERTopicModel import BERTopicModel
from bertrend.utils.data_loading import (
    DOCUMENT_ID_COLUMN, SOURCE_COLUMN, TEXT_COLUMN, TIMESTAMP_COLUMN, URL_COLUMN, group_by_days,
)

# --- burst window (0.11: Dec 2022 keyword ramp) ---
DATE_START = pd.Timestamp("2022-11-01")
DATE_END = pd.Timestamp("2023-01-31")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
INCEPTION = pd.Timestamp("2023-05-17")
HEADLINES_CACHE = OUTPUT_DIR / "bloomberg_headlines_2018_2023.parquet"
EMB_CACHE = OUTPUT_DIR / "genai_burst_emb_2022nov_2023jan.npy"
META_CACHE = OUTPUT_DIR / "genai_burst_meta.parquet"

BLOOMBERG_WIRES = ["BN", "BFW", "BBO"]
EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42

GRANULARITIES = [7, 14]
PRIMARY_G = 7
MIN_TOPIC, MIN_SAMPLES = 8, 3          # finer — small burst corpus
MIN_SIMILARITY = 0.65
MIN_ACTIVE_SLICES = 3                  # ~3 months → max ~13 slices @ 7d
VECTORIZER_MIN_DF = 1
POOL_CAP = 80_000                      # stratified cap if corpus > this

# 0.11 STANDARD lexicon (validation / scoped subcorpus only)
GENAI_T1 = [
    r"generative ai", r"generative artificial intelligence",
    r"large language model", r"large language models", r"\bllms\b",
    r"chatgpt", r"gpt-3\.5", r"gpt-3", r"gpt-4",
    r"foundation model", r"foundation models",
    r"stable diffusion", r"midjourney", r"dall-e", r"dalle", r"text-to-image", r"text to image",
]
GENAI_T2 = [
    r"\bopenai\b", r"\banthropic\b", r"chatgpt-like", r"chatgpt-style",
    r"prompt engineering", r"ai chatbot", r"ai chat bot",
]
GENAI_STANDARD = re.compile("|".join(f"(?:{p})" for p in GENAI_T1 + GENAI_T2), re.I)
GENAI_STRICT = re.compile("|".join(f"(?:{p})" for p in GENAI_T1), re.I)

print(f"Device {DEVICE} | burst {DATE_START.date()}→{DATE_END.date()} | granularities {GRANULARITIES}d")

Device mps | burst 2022-11-01→2023-01-31 | granularities [7, 14]d


## 1. Load burst-window headlines

Slice from `bloomberg_headlines_2018_2023.parquet` ([`0.11`](0.11-genai-keyword-timeline.ipynb) §8).

In [2]:
if not HEADLINES_CACHE.exists():
    raise FileNotFoundError(f"Run 0.11 §8 first — missing {HEADLINES_CACHE.name}")

news = pd.read_parquet(HEADLINES_CACHE)
news["date"] = pd.to_datetime(news["date"])
news = news[(news.date >= DATE_START) & (news.date <= DATE_END)].reset_index(drop=True)
news["hl"] = news["Headline"].str.lower()
news["lex_hit"] = news["hl"].str.contains(GENAI_STANDARD, na=False)
news["strict_hit"] = news["hl"].str.contains(GENAI_STRICT, na=False)

print(f"Burst corpus: {len(news):,} headlines ({news.date.min().date()} → {news.date.max().date()})")
print(f"  STANDARD lexicon hits: {news.lex_hit.sum():,} ({news.lex_hit.mean()*100:.3f}%)")
print(f"  STRICT lexicon hits:   {news.strict_hit.sum():,}")
by_m = news.groupby(news.date.dt.to_period("M")).agg(
    total=("Headline", "count"), lex=("lex_hit", "sum"), strict=("strict_hit", "sum"))
by_m["share_bp"] = (by_m["lex"] / by_m["total"] * 10_000).round(2)
print("\nMonthly keyword ground truth:")
print(by_m.to_string())

Burst corpus: 213,056 headlines (2022-11-01 → 2023-01-30)
  STANDARD lexicon hits: 94 (0.044%)
  STRICT lexicon hits:   53

Monthly keyword ground truth:
         total  lex  strict  share_bp
date                                 
2022-11  91315    0       0      0.00
2022-12  56376   16       7      2.84
2023-01  65365   78      46     11.93


## 2. Helpers + embed (cached)

In [6]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

CUSTOM_STOP = list(ENGLISH_STOP_WORDS.union({
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock",
    "stocks", "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
}))


def make_df(sub: pd.DataFrame) -> pd.DataFrame:
    d = pd.DataFrame({TEXT_COLUMN: sub["Headline"].values,
                      TIMESTAMP_COLUMN: pd.to_datetime(sub["date"].values)})
    d[DOCUMENT_ID_COLUMN] = range(len(d))
    d[SOURCE_COLUMN] = "bloomberg"
    d[URL_COLUMN] = None
    return d.reset_index(drop=True)


def embed(texts_or_df, show_progress=True) -> np.ndarray:
    texts = texts_or_df[TEXT_COLUMN].tolist() if isinstance(texts_or_df, pd.DataFrame) else texts_or_df
    return embedder.encode(texts, batch_size=64, show_progress_bar=show_progress,
                           convert_to_numpy=True, normalize_embeddings=True)


def cap_stratified(sub: pd.DataFrame, cap: int) -> pd.DataFrame:
    if len(sub) <= cap:
        return sub
    per_day = max(1, cap // sub["date"].dt.normalize().nunique())
    return (sub.groupby(sub["date"].dt.normalize(), group_keys=False)
              .apply(lambda g: g.sample(min(len(g), per_day), random_state=RANDOM_SEED)))


def _bertopic(min_topic_size, min_samples, min_df=VECTORIZER_MIN_DF):
    cfg = f"""
[global]
language = "English"
[bertopic_model]
top_n_words = 10
verbose = false
representation_model = ["MaximalMarginalRelevance"]
zeroshot_topic_list = []
zeroshot_min_similarity = 0
[umap_model]
n_neighbors = 15
n_components = 5
min_dist = 0.0
metric = "cosine"
random_state = {RANDOM_SEED}
[hdbscan_model]
min_cluster_size = {min_topic_size}
min_samples = {min_samples}
metric = "euclidean"
cluster_selection_method = "eom"
prediction_data = true
[vectorizer_model]
ngram_range = [1, 1]
stop_words = true
min_df = {min_df}
[ctfidf_model]
bm25_weighting = false
reduce_frequent_words = true
[mmr_model]
diversity = 0.3
[reduce_outliers]
strategy = "c-tf-idf"
"""
    tm = BERTopicModel(cfg)
    tm.vectorizer_model = CountVectorizer(stop_words=CUSTOM_STOP,
                                          token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
                                          ngram_range=(1, 2), min_df=min_df)
    tm.config["bertopic_model"]["representation_model"] = [MaximalMarginalRelevance(diversity=0.4)]
    return tm


def run_bertrend(df, embeddings, granularity, tag="burst"):
    bt = BERTrend(topic_model=_bertopic(MIN_TOPIC, MIN_SAMPLES))
    bt.config["granularity"] = granularity
    bt.config["min_similarity"] = MIN_SIMILARITY
    grouped = {ts: g for ts, g in group_by_days(df=df, day_granularity=granularity).items() if not g.empty}
    bt.train_topic_models(grouped_data=grouped, embedding_model=embedder, embeddings=embeddings,
                          bertrend_models_path=OUTPUT_DIR / f"_genai_{tag}", save_topic_models=False)
    if bt.merged_df is None:
        return None, len(grouped)
    bt.calculate_signal_popularity()
    return bt, len(grouped)


def theme_table(bt) -> pd.DataFrame:
    rep = {}
    for _, r in bt.merged_df.drop_duplicates("Topic").iterrows():
        x = r.get("Representation")
        rep[int(r["Topic"])] = ", ".join(x[:8]) if isinstance(x, (list, tuple)) else str(x)
    rows = []
    for tid, d in bt.topic_sizes.items():
        st = pd.to_datetime(list(d.get("Timestamps", [])))
        if len(st) == 0:
            continue
        rows.append({"theme_id": int(tid), "slices": int(st.normalize().nunique()),
                     "first_seen": st.min().normalize(), "last_seen": st.max().normalize(),
                     "docs": int(max(d.get("Docs_Count", [0]) or [0])),
                     "keywords": rep.get(int(tid), "")})
    return pd.DataFrame(rows).sort_values(["slices", "docs"], ascending=False).reset_index(drop=True)


def _flatten_docs(documents) -> list[str]:
    """merged_df['Documents']: strings and/or (timestamp, [headlines]) tuples."""
    out = []
    if not isinstance(documents, (list, tuple)):
        return out
    for item in documents:
        if isinstance(item, str):
            out.append(item)
        elif isinstance(item, tuple) and len(item) == 2 and isinstance(item[1], list):
            out.extend(t for t in item[1] if isinstance(t, str))
        elif isinstance(item, list):
            out.extend(t for t in item if isinstance(t, str))
    return out


def label_genai_overlap(bt) -> pd.DataFrame:
    """Post-hoc: fraction of each theme's merged documents matching STRICT / STANDARD lexicon."""
    docs_by = dict(zip(bt.merged_df["Topic"], bt.merged_df["Documents"]))
    rows = []
    for tid in bt.merged_df["Topic"].dropna().unique():
        tid = int(tid)
        docs = _flatten_docs(docs_by.get(tid, []))
        if not docs:
            continue
        hl = pd.Series(d.lower() for d in docs)
        rows.append({
            "theme_id": tid,
            "n_docs": len(docs),
            "pct_strict": round(hl.str.contains(GENAI_STRICT, na=False).mean() * 100, 1),
            "pct_standard": round(hl.str.contains(GENAI_STANDARD, na=False).mean() * 100, 1),
        })
    return pd.DataFrame(rows)


def _slice_totals(df, granularity):
    return pd.Series({
        pd.Timestamp(ts).normalize(): len(g)
        for ts, g in group_by_days(df=df, day_granularity=granularity).items() if not g.empty
    })


def theme_intensity(bt, df, theme_id, granularity):
    slice_totals = _slice_totals(df, granularity)
    data = bt.topic_sizes.get(int(theme_id), {})
    docs, ts_list = data.get("Docs_Count", []), data.get("Timestamps", [])
    cum = {}
    for i, ts in enumerate(ts_list):
        cum[pd.Timestamp(ts).normalize()] = float(docs[i] if i < len(docs) else 0)
    rows, prev = [], 0.0
    for ts in sorted(cum):
        new = max(cum[ts] - prev, 0)
        total = float(slice_totals.get(ts, np.nan))
        share = new / total if total else np.nan
        rows.append({"timestamp": ts, "new_docs": new, "slice_total": total,
                       "share_per10k": 10_000 * share if pd.notna(share) else np.nan})
        prev = cum[ts]
    return pd.DataFrame(rows)


def print_themes(label, t, n_slices, corpus_tag):
    pre = t[t.slices >= MIN_ACTIVE_SLICES]
    print(f"\n{'=' * 72}\n{label} · {corpus_tag} · {n_slices} slices · {len(t)} themes · stable(≥{MIN_ACTIVE_SLICES})={len(pre)}")
    for _, r in t.head(12).iterrows():
        flag = "★" if r.slices >= MIN_ACTIVE_SLICES else " "
        print(f"  [{flag}] T{int(r.theme_id):>3} {int(r.slices):>2}s  "
              f"{r['first_seen'].date()}  {r.keywords[:58]}")
    return pre

print("helpers ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

helpers ready


## 3. Prepare corpora + embeddings

In [7]:
# --- all-news (stratified cap) ---
pool_all = cap_stratified(news, POOL_CAP).sort_values("date").reset_index(drop=True)
df_all = make_df(pool_all)
pool_all = pool_all.reset_index(drop=True)
print(f"All-news pool: {len(pool_all):,}")

# --- lexicon-scoped (validation subcorpus) ---
pool_lex = news[news["lex_hit"]].sort_values("date").reset_index(drop=True)
df_lex = make_df(pool_lex)
print(f"Lexicon-scoped pool: {len(pool_lex):,}")

if EMB_CACHE.exists() and len(pool_all) == np.load(EMB_CACHE, mmap_mode="r").shape[0]:
    emb_all = np.load(EMB_CACHE)
    print(f"Loaded embed cache {emb_all.shape}")
else:
    print("Embedding all-news pool…")
    emb_all = embed(df_all)
    np.save(EMB_CACHE, emb_all)
    meta = pool_all[["Headline", "date", "lex_hit", "strict_hit"]].assign(corpus="all")
    meta.to_parquet(META_CACHE, index=False)
    print(f"Wrote {EMB_CACHE.name} + {META_CACHE.name}")

EMB_LEX_CACHE = OUTPUT_DIR / "genai_burst_emb_lex.npy"
if len(pool_lex) >= MIN_TOPIC:
    if EMB_LEX_CACHE.exists():
        emb_lex = np.load(EMB_LEX_CACHE)
        print(f"Loaded lex embed {emb_lex.shape}")
    else:
        print("Embedding lexicon pool…")
        emb_lex = embed(df_lex)
        np.save(EMB_LEX_CACHE, emb_lex)
else:
    emb_lex = None
    print("Lexicon pool too small for BERTrend")

All-news pool: 62,002
Lexicon-scoped pool: 94
Loaded embed cache (62002, 768)
Loaded lex embed (94, 768)


## 4. BERTrend — all-news + lexicon-scoped

In [8]:
all_results = {}
lex_results = {}

for g in GRANULARITIES:
    bt, n_sl = run_bertrend(df_all, emb_all, g, tag=f"burst_all_{g}d")
    if bt is None:
        print(f"\n[{g}d all-news] no merged themes ({n_sl} slices)")
        continue
    t = theme_table(bt)
    ov = label_genai_overlap(bt)
    t = t.merge(ov, on="theme_id", how="left")
    pre = print_themes(f"ALL-NEWS @ {g}d", t, n_sl, "all")
    out = t.assign(corpus="all", granularity_days=g, n_slices=n_sl)
    path = OUTPUT_DIR / f"scan_genai_burst_all_{g}d.parquet"
    out.to_parquet(path, index=False)
    all_results[g] = {"bt": bt, "table": t, "path": path, "df": df_all, "raw": pool_all, "emb": emb_all}
    print(f"  → {path.name}")
    genai_themes = t[t["pct_standard"].fillna(0) >= 20]
    if not genai_themes.empty:
        print(f"  genAI-like themes (≥20% STANDARD overlap): {genai_themes.theme_id.tolist()}")

    if emb_lex is not None and len(pool_lex) >= 30:
        bt2, n_sl2 = run_bertrend(df_lex, emb_lex, g, tag=f"burst_lex_{g}d")
        if bt2 is None:
            print(f"\n[{g}d lexicon] no merged themes")
            continue
        t2 = theme_table(bt2)
        ov2 = label_genai_overlap(bt2)
        t2 = t2.merge(ov2, on="theme_id", how="left")
        print_themes(f"LEXICON-SCOPED @ {g}d", t2, n_sl2, "lex")
        out2 = t2.assign(corpus="lex", granularity_days=g, n_slices=n_sl2)
        path2 = OUTPUT_DIR / f"scan_genai_burst_lex_{g}d.parquet"
        out2.to_parquet(path2, index=False)
        lex_results[g] = {"bt": bt2, "table": t2, "path": path2, "df": df_lex, "raw": pool_lex, "emb": emb_lex}
        print(f"  → {path2.name}")

2026-06-15 16:26:50.623 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 1/13...
2026-06-15 16:26:50.624 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2022-11-01 00:00:00
2026-06-15 16:26:50.624 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 5006
2026-06-15 16:26:50.624 | DEBUG    | bertrend.BERTrend:_train_by_period:189 - Creating topic model...
2026-06-15 16:26:50.624 | DEBUG    | bertrend.BERTopicModel:fit:263 - 	Initializing BERTopic model
2026-06-15 16:26:50.625 | SUCCESS  | bertrend.BERTopicModel:fit:273 - 	BERTopic model instance created successfully
2026-06-15 16:26:50.625 | DEBUG    | bertrend.BERTopicModel:fit:275 - 	Fitting BERTopic model
2026-06-15 16:26:54.351 | DEBUG    | bertrend.BERTopicModel:fit:282 - 	Reducing outliers
2026-06-15 16:26:54.361 | DEBUG    | bertrend.BERTopicModel:fit:289 - 	Updating topics


2026-06-15 16:26:54,361 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-15 16:26:54.414 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 16:26:54.414 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 16:26:54.424 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2022-11-01 00:00:00
2026-06-15 16:26:54.424 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 2/13...
2026-06-15 16:26:54.425 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2022-11-08 00:00:00
2026-06-15 16:26:54.425 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 4960
2026-06-15 16:26:54.425 | DEBUG    | bertrend.BERTrend:_train_by_period:189 - Creating topic model...
2026-06-15 16:26:54.425 | DEBUG    | bertrend.BERTopicModel:fit:263 - 	Initializing BERTopic model
2026-06-15 16:26:54.426 | SUCCESS  | bertrend.BERTopicModel:fit:273 - 	BERTopic model instance created successfully
2026-

2026-06-15 16:26:58,085 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-15 16:26:58.137 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 16:26:58.137 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 16:26:58.145 | DEBUG    | bertrend.BERTrend:merge_models_with:360 - Merging topic models for timestamps: 2022-11-01 00:00:00 and 2022-11-08 00:00:00
2026-06-15 16:26:58.380 | SUCCESS  | bertrend.BERTrend:merge_models_with:428 - Models 2022-11-08 00:00:00 merged successfully with others
2026-06-15 16:26:58.381 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2022-11-08 00:00:00
2026-06-15 16:26:58.381 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 3/13...
2026-06-15 16:26:58.382 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2022-11-15 00:00:00
2026-06-15 16:26:58.382 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 4815
2026-06-15 16:26:58.382 

...

2026-06-15 16:28:30.922 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 16:28:30.922 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 16:28:30.926 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2022-12-29 00:00:00
2026-06-15 16:28:30.926 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 4/5...
2026-06-15 16:28:30.926 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2023-01-12 00:00:00
2026-06-15 16:28:30.927 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 31
2026-06-15 16:28:30.927 | DEBUG    | bertrend.BERTrend:_train_by_period:189 - Creating topic model...
2026-06-15 16:28:30.927 | DEBUG    | bertrend.BERTopicModel:fit:263 - 	Initializing BERTopic model
2026-06-15 16:28:30.927 | SUCCESS  | bertrend.BERTopicModel:fit:273 - 	BERTopic model instance created successfully
2026-06-

2026-06-15 16:28:30,959 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-15 16:28:30.962 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 16:28:30.962 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 16:28:30.967 | DEBUG    | bertrend.BERTrend:merge_models_with:360 - Merging topic models for timestamps: 2022-12-29 00:00:00 and 2023-01-12 00:00:00
2026-06-15 16:28:30.970 | SUCCESS  | bertrend.BERTrend:merge_models_with:428 - Models 2023-01-12 00:00:00 merged successfully with others
2026-06-15 16:28:30.970 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2023-01-12 00:00:00
2026-06-15 16:28:30.970 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 5/5...
2026-06-15 16:28:30.971 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2023-01-26 00:00:00
2026-06-15 16:28:30.971 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 24
2026-06-15 16:28:30.971 | D

2026-06-15 16:28:30,998 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-15 16:28:31.001 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 16:28:31.001 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 16:28:31.005 | DEBUG    | bertrend.BERTrend:merge_models_with:360 - Merging topic models for timestamps: 2023-01-12 00:00:00 and 2023-01-26 00:00:00
2026-06-15 16:28:31.008 | SUCCESS  | bertrend.BERTrend:merge_models_with:428 - Models 2023-01-26 00:00:00 merged successfully with others
2026-06-15 16:28:31.009 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2023-01-26 00:00:00
2026-06-15 16:28:31.009 | ERROR    | bertrend.BERTrend:train_topic_models:323 - Training completed with failures for 2 periods: [Timestamp('2022-12-01 00:00:00'), Timestamp('2022-12-15 00:00:00')]
2026-06-15 16:28:31.009 | WARNING  | bertrend.BERTrend:train_topic_models:327 - Successfully trained topic models for 3 periods: [Timestamp('2022-12-29 00:00

## 5. Compare + intensity (primary granularity)

In [10]:
rows = []
intensity_parts = []

for corpus, res_dict in [("all", all_results), ("lex", lex_results)]:
    if PRIMARY_G not in res_dict:
        continue
    t = res_dict[PRIMARY_G]["table"]
    best = t.sort_values(["pct_standard", "slices"], ascending=[False, False]).head(1)
    rows.append({
        "corpus": corpus, "granularity": PRIMARY_G, "n_themes": len(t),
        "n_stable": int((t.slices >= MIN_ACTIVE_SLICES).sum()),
        "best_theme": int(best.theme_id.iloc[0]) if len(best) else None,
        "best_pct_standard": best.pct_standard.iloc[0] if len(best) else None,
        "best_keywords": best.keywords.iloc[0][:60] if len(best) else "",
    })
    if len(best):
        tid = int(best.theme_id.iloc[0])
        bt = res_dict[PRIMARY_G]["bt"]
        df = res_dict[PRIMARY_G]["df"]
        inten = theme_intensity(bt, df, tid, PRIMARY_G)
        inten = inten.assign(corpus=corpus, theme_id=tid, granularity_days=PRIMARY_G)
        intensity_parts.append(inten)

compare = pd.DataFrame(rows)
if not compare.empty:
    compare.to_parquet(OUTPUT_DIR / "genai_burst_compare.parquet", index=False)
    print(compare.to_string(index=False))

if intensity_parts:
    intensity = pd.concat(intensity_parts, ignore_index=True)
    intensity.to_parquet(OUTPUT_DIR / "genai_burst_intensity.parquet", index=False)
    print("\nShare intensity (best genAI-overlap theme @ 7d):")
    print(intensity.to_string(index=False))

    fig = go.Figure()
    for corpus in intensity.corpus.unique():
        sub = intensity[intensity.corpus == corpus]
        fig.add_trace(go.Scatter(x=sub.timestamp, y=sub.share_per10k, mode="lines+markers", name=corpus))
    fig.add_vline(x=CHATGPT_LAUNCH, line_dash="dash", line_color="gray")
    fig.update_layout(title=f"Best genAI-overlap theme intensity (@ {PRIMARY_G}d slices)",
                      xaxis_title="Slice", yaxis_title="Share (per 10k headlines)", height=420)
    fig.write_html(OUTPUT_DIR / "genai_burst_intensity.html")
    fig.show()

# rep headlines for best all-news genAI theme
if PRIMARY_G in all_results:
    t = all_results[PRIMARY_G]["table"].sort_values("pct_standard", ascending=False)
    if len(t) and t.iloc[0]["pct_standard"] > 0:
        tid = int(t.iloc[0].theme_id)
        bt = all_results[PRIMARY_G]["bt"]
        sub = bt.merged_df[bt.merged_df["Topic"] == tid]
        if not sub.empty:
            c = np.asarray(sub.iloc[0]["Embedding"], float)
            c /= np.linalg.norm(c) + 1e-12
            idx = np.argsort(-(all_results[PRIMARY_G]["emb"] @ c))[:5]
            print(f"\nRep headlines — all-news T{tid} ({t.iloc[0].keywords[:50]}):")
            for i, ix in enumerate(idx, 1):
                print(f"  {i}. {pool_all.iloc[ix].Headline[:95]}")

corpus  granularity  n_themes  n_stable  best_theme  best_pct_standard                                                best_keywords
   all            7       174       174         146                1.2 reject, faa, pilot, air taxi, taxi, united, dispute, contrac
   lex            7         1         0           0              100.0 chatgpt, microsoft boost, boost, boost investment, tech, mic

Share intensity (best genAI-overlap theme @ 7d):
 timestamp  new_docs  slice_total  share_per10k corpus  theme_id  granularity_days
2022-11-01      12.0       5006.0     23.971235    all       146                 7
2022-11-08      13.0       4960.0     26.209677    all       146                 7
2022-11-15       0.0       4815.0      0.000000    all       146                 7
2022-11-22       0.0       4787.0      0.000000    all       146                 7
2022-11-29       0.0       4917.0      0.000000    all       146                 7
2022-12-20      22.0       4276.0     51.449953    all  


Rep headlines — all-news T146 (reject, faa, pilot, air taxi, taxi, united, disput):
  1. *UNITED COMMENTS ON PILOT NEGOTIATIONS IN EMAILED STATEMENT
  2. American Air’s Pilot Union Leaders Reject Proposed New Contract
  3. Pilot Union Merger Proposed Amid Labor Talks at US Airlines (1)
  4. *UNITED PILOTS REJECT TENTATIVE CONTRACT AGREEMENT: CNBC
  5. *JETBLUE PILOTS CALL FOR CONTRACT AS NEGOTIATIONS CONTINUE


## 6. Quick reload

In [ ]:
for pat in sorted(OUTPUT_DIR.glob("scan_genai_burst_*.parquet")):
    t = pd.read_parquet(pat)
    print(f"\n{pat.name}: {len(t)} themes")
    cols = [c for c in ["theme_id", "slices", "first_seen", "pct_standard", "pct_strict", "keywords"] if c in t.columns]
    print(t[cols].head(8).to_string(index=False))